In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from datasets import load_dataset, Image as HFImage, DatasetDict
import json
import pandas as pd
import matplotlib.pyplot as plt

# Step 1. Load data

In [ ]:
dataset = load_dataset("Codatta/MM-Food-100K")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

MM-Food-100K.csv:   0%|          | 0.00/28.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 100000
    })
})


# Step 2. Split the dataset into training/validation/test sets

In [ ]:
train_test_split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_val_split = train_test_split['train'].train_test_split(test_size=0.11, seed=42)

# merge back into a single DatasetDict
dataset = DatasetDict({
    'train': train_val_split['train'],
    'validation': train_val_split['test'],
    'test': train_test_split['test']
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 80100
    })
    validation: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 9900
    })
    test: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 10000
    })
})


# Step 3. Cast calories data to float and add to the dataset

In [ ]:
def get_calories(example):
    profile = example['nutritional_profile']
    # if it's a string, parse it to a dict
    if isinstance(profile, str):
        try:
            profile = json.loads(profile.replace("'", '"'))
        except:
            return {'calories': None}

    # extract calories_kcal
    calories = profile.get('calories_kcal')
    return {'calories': float(calories) if calories is not None else None}

# add a 'calories' column to the dataset
dataset = dataset.map(get_calories)

# print dataset's features
print(dataset['train'].features)
print(dataset['validation'].features)
print(dataset['test'].features)

# print the calories data of the first training image
print(f"The first calorie value: {dataset['train'][0]['calories']} kcal")

Map:   0%|          | 0/80100 [00:00<?, ? examples/s]

Map:   0%|          | 0/9900 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

{'image_url': Value('string'), 'camera_or_phone_prob': Value('float64'), 'food_prob': Value('float64'), 'dish_name': Value('string'), 'food_type': Value('string'), 'ingredients': Value('string'), 'portion_size': Value('string'), 'nutritional_profile': Value('string'), 'cooking_method': Value('string'), 'sub_dt': Value('int64'), 'calories': Value('float64')}
{'image_url': Value('string'), 'camera_or_phone_prob': Value('float64'), 'food_prob': Value('float64'), 'dish_name': Value('string'), 'food_type': Value('string'), 'ingredients': Value('string'), 'portion_size': Value('string'), 'nutritional_profile': Value('string'), 'cooking_method': Value('string'), 'sub_dt': Value('int64'), 'calories': Value('float64')}
{'image_url': Value('string'), 'camera_or_phone_prob': Value('float64'), 'food_prob': Value('float64'), 'dish_name': Value('string'), 'food_type': Value('string'), 'ingredients': Value('string'), 'portion_size': Value('string'), 'nutritional_profile': Value('string'), 'cooking_me

# Step 4. Process images and create DataLoaders

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
import torch

# define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

MAX_CALORIES = 3500.0

def custom_transform(examples):
    # process images
    examples['pixel_values'] = [transform(img.convert("RGB")) for img in examples['image_url']]

    # normalize calories
    examples['labels'] = [float(cal) / MAX_CALORIES for cal in examples['calories']]

    return examples

# apply the transform
dataset.set_transform(custom_transform)

# Hugging Face -> PyTorch DataLoader
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

# create the DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(dataset['train'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
val_loader = DataLoader(dataset['validation'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset['test'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Step 5. Load weights of the pre-trained ResNet152 on Food2K

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
# initialize the standard architecture
model = models.resnet152(weights=None)

# change the 'fc' layer to 2000 to satisfy the .pth file (pre-trained model on Food2K dataset)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2000)

# load the weights from pre-trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_dict = torch.load('food2k_resnet152_0.0001.pth', map_location=device)

# clean 'module.' prefix if it exists from DataParallel training
new_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

model.load_state_dict(new_state_dict)
model.to(device)

# strip the last layer to create feature extractor
modules = list(model.children())[:-1]
resnet152_features = nn.Sequential(*modules)

resnet152_features.to(device)
resnet152_features.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


# Step 6. Feature Extraction

In [ ]:
import os
import time
from PIL import Image, ImageFile

save_dir_features = "/content/drive/MyDrive/MIE1517/Project/resnet152_features"
os.makedirs(save_dir_features, exist_ok=True)

def save_resnet152_features(dataloader, loadername):
    print(f'Starting feature extraction for: {loadername}...')

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            img = batch['pixel_values'].to(device)
            label = batch['labels'].to(device)

            features = resnet152_features(img)
            features = features.view(features.size(0), -1)

            file_path = os.path.join(save_dir_features, f'{loadername}_{i}.pt')
            torch.save((features.cpu(), label.cpu()), file_path)

            if i % 100 == 0:
                print(f"Batch {i} saved...")

save_resnet152_features(train_loader, 'train')
save_resnet152_features(val_loader, 'val')
save_resnet152_features(test_loader, 'test')